# 05_agente_consultas

## Objetivo
Construir un panel interactivo dentro de Jupyter que permita consultar el ranking, explicar departamentos y simular cambios en los pesos del índice de oportunidad.

## Alcance
- Cargar los resultados del índice
- Crear funciones de consulta
- Construir un panel interactivo con widgets
- Permitir consultas guiadas dentro del notebook


## 1. Librerías

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

## 2. Rutas y carga de datos

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'

df = pd.read_csv(DATA_PROCESSED / 'ranking_oportunidad.csv')
df.head()

## 3. Preparación de datos

In [ ]:
if 'nivel' not in df.columns:
    df['nivel'] = pd.qcut(df['indice_oportunidad'], q=3, labels=['Bajo', 'Medio', 'Alto'])

ranking = df.sort_values(by='indice_oportunidad', ascending=False).reset_index(drop=True)
departamentos = sorted(ranking['departamento'].dropna().unique().tolist())

ranking[['departamento', 'indice_oportunidad', 'nivel']].head()

## 4. Funciones del agente

Estas funciones representan las capacidades básicas del agente.

In [ ]:
def get_top_departamentos(n=10):
    return ranking[['departamento', 'indice_oportunidad', 'nivel']].head(n)


def explicar_departamento(departamento):
    fila = ranking[ranking['departamento'] == departamento].copy()
    if fila.empty:
        return f'No se encontró información para {departamento}.'
    fila = fila.iloc[0]
    texto = f"""
**Departamento:** {fila['departamento']}  
**Índice de oportunidad:** {fila['indice_oportunidad']:.3f}  
**Nivel:** {fila['nivel']}  

Este departamento presenta un comportamiento que combina necesidad económica, condiciones de inclusión financiera y viabilidad digital. El valor del índice resume el efecto conjunto de las variables consideradas en el modelo.
"""
    return texto


def recalcular_indice(peso_pobreza, peso_micro, peso_productos, peso_atm, peso_internet):
    total = peso_pobreza + peso_micro + peso_productos + peso_atm + peso_internet
    if total == 0:
        raise ValueError('La suma de los pesos no puede ser cero.')
    pesos = {
        'pobreza_n': peso_pobreza / total,
        'microcredito_n': peso_micro / total,
        'productos_n': peso_productos / total,
        'atm_n': peso_atm / total,
        'internet_n': peso_internet / total,
    }
    temp = df.copy()
    temp['indice_simulado'] = (
        temp['pobreza_n'] * pesos['pobreza_n'] +
        temp['microcredito_n'] * pesos['microcredito_n'] +
        temp['productos_n'] * pesos['productos_n'] +
        temp['atm_n'] * pesos['atm_n'] +
        temp['internet_n'] * pesos['internet_n']
    )
    temp = temp.sort_values(by='indice_simulado', ascending=False).reset_index(drop=True)
    return temp[['departamento', 'indice_simulado']].head(10)


## 5. Controles del panel

In [ ]:
consulta_dropdown = widgets.Dropdown(
    options=['Ranking general', 'Explicar departamento', 'Simular pesos'],
    value='Ranking general',
    description='Consulta:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

departamento_dropdown = widgets.Dropdown(
    options=departamentos,
    description='Departamento:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='350px')
)

peso_pobreza = widgets.FloatSlider(value=0.30, min=0, max=1, step=0.05, description='Pobreza')
peso_micro = widgets.FloatSlider(value=0.30, min=0, max=1, step=0.05, description='Microcrédito')
peso_productos = widgets.FloatSlider(value=0.15, min=0, max=1, step=0.05, description='Productos')
peso_atm = widgets.FloatSlider(value=0.10, min=0, max=1, step=0.05, description='ATM')
peso_internet = widgets.FloatSlider(value=0.15, min=0, max=1, step=0.05, description='Internet')

boton = widgets.Button(description='Generar', button_style='primary')
salida = widgets.Output()

## 6. Lógica del panel

In [ ]:
def ejecutar_consulta(_):
    with salida:
        clear_output()
        consulta = consulta_dropdown.value
        
        if consulta == 'Ranking general':
            display(Markdown('### Top 10 departamentos'))
            display(get_top_departamentos(10))
        
        elif consulta == 'Explicar departamento':
            display(Markdown('### Explicación del departamento'))
            display(Markdown(explicar_departamento(departamento_dropdown.value)))
        
        elif consulta == 'Simular pesos':
            display(Markdown('### Top 10 con pesos simulados'))
            display(recalcular_indice(
                peso_pobreza.value,
                peso_micro.value,
                peso_productos.value,
                peso_atm.value,
                peso_internet.value
            ))

boton.on_click(ejecutar_consulta)

## 7. Panel interactivo

In [ ]:
panel = widgets.VBox([
    consulta_dropdown,
    departamento_dropdown,
    peso_pobreza,
    peso_micro,
    peso_productos,
    peso_atm,
    peso_internet,
    boton,
    salida
])

display(panel)